In [33]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Data Import

In [34]:
# ============================================================
# CONFIG
# ============================================================

RAW_DATA_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\Data\highD-dataset-v1.0\data"
)

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

OUTPUT_DIR = PROJECT_DIR / "processed_data_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_RECORDINGS = 40
RANDOM_SEED = 42

MIN_EPISODE_SECONDS = 5.0
DEFAULT_FRAMERATE = 25

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

print("Raw data dir:", RAW_DATA_DIR)
print("Output dir:", OUTPUT_DIR)

assert RAW_DATA_DIR.exists(), f"Raw data directory not found: {RAW_DATA_DIR}"
assert OUTPUT_DIR.exists(), f"Output directory not found: {OUTPUT_DIR}"

Raw data dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\Data\highD-dataset-v1.0\data
Output dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final


In [35]:

# -------------------------------------------------
# DATA DIRECTORY
# -------------------------------------------------
DATA_DIR = RAW_DATA_DIR

# -------------------------------------------------
# FIND ALL TRACK FILES AND SELECT 40 RECORDINGS
# -------------------------------------------------
track_files = sorted(DATA_DIR.glob("*_tracks.csv"))

# Extract recording IDs like 01, 02, 03...
recording_ids = [f.stem.split("_")[0] for f in track_files]

# Keep the first N recordings
selected_ids = recording_ids[:N_RECORDINGS]

print("Selected recordings:")
for rec_id in selected_ids:
    print("-", rec_id)

# -------------------------------------------------
# LOAD TRACKS, TRACKSMETA, AND RECORDINGMETA
# -------------------------------------------------
tracks_list = []
tracks_meta_list = []
recording_meta_list = []

for rec_id in selected_ids:
    tracks_file = DATA_DIR / f"{rec_id}_tracks.csv"
    tracks_meta_file = DATA_DIR / f"{rec_id}_tracksMeta.csv"
    recording_meta_file = DATA_DIR / f"{rec_id}_recordingMeta.csv"

    tracks = pd.read_csv(tracks_file)
    tracks_meta = pd.read_csv(tracks_meta_file)
    recording_meta = pd.read_csv(recording_meta_file)

    # Add source info to tracks
    tracks["recording_id"] = rec_id
    tracks["source_file"] = tracks_file.name
    tracks["unique_id"] = tracks["recording_id"].astype(str) + "_" + tracks["id"].astype(str)

    # Add source info to tracksMeta
    tracks_meta["recording_id"] = rec_id
    tracks_meta["source_file"] = tracks_meta_file.name
    tracks_meta["unique_id"] = tracks_meta["recording_id"].astype(str) + "_" + tracks_meta["id"].astype(str)

    # Add source info to recordingMeta
    recording_meta["recording_id"] = rec_id
    recording_meta["source_file"] = recording_meta_file.name

    tracks_list.append(tracks)
    tracks_meta_list.append(tracks_meta)
    recording_meta_list.append(recording_meta)

# -------------------------------------------------
# COMBINE EVERYTHING
# -------------------------------------------------
tracks = pd.concat(tracks_list, ignore_index=True)
tracks_meta = pd.concat(tracks_meta_list, ignore_index=True)
recording_meta = pd.concat(recording_meta_list, ignore_index=True)

# -------------------------------------------------
# CHECK RESULTS
# -------------------------------------------------
print("\n=== FILES LOADED SUCCESSFULLY ===")
print("Number of selected recordings:", len(selected_ids))
print("Tracks shape:", tracks.shape)
print("TracksMeta shape:", tracks_meta.shape)
print("RecordingMeta shape:", recording_meta.shape)

print("\n=== TRACK FILES USED ===")
for rec_id in selected_ids:
    print(f"{rec_id}_tracks.csv")

print("\n=== TRACKSMETA FILES USED ===")
for rec_id in selected_ids:
    print(f"{rec_id}_tracksMeta.csv")

print("\n=== RECORDINGMETA FILES USED ===")
for rec_id in selected_ids:
    print(f"{rec_id}_recordingMeta.csv")

# -------------------------------------------------
# BASIC STATISTICS
# -------------------------------------------------
print("\n=== BASIC INFO ===")
print(f"Number of rows (observations): {len(tracks)}")
print(f"Number of vehicles: {tracks['unique_id'].nunique()}")
print(f"Number of lanes: {tracks['laneId'].nunique()}")
print(f"Frame range: {tracks['frame'].min()} → {tracks['frame'].max()}")

print("\n=== VEHICLE TYPES ===")
print(tracks_meta["class"].value_counts())

print("\n=== LANE USAGE ===")
print(tracks["laneId"].value_counts().sort_index())

# -------------------------------------------------
# LANE STABILITY
# -------------------------------------------------
lane_counts = tracks.groupby("unique_id")["laneId"].nunique()

num_lane_stable = (lane_counts == 1).sum()
num_lane_changers = (lane_counts > 1).sum()

print("\n=== LANE STABILITY ===")
print(f"Vehicles total: {len(lane_counts)}")
print(f"Lane-stable vehicles: {num_lane_stable}")
print(f"Lane-changing vehicles: {num_lane_changers}")
print(f"Fraction lane-changing: {num_lane_changers / len(lane_counts):.2%}")

Selected recordings:
- 01
- 02
- 03
- 04
- 05
- 06
- 07
- 08
- 09
- 10
- 11
- 12
- 13
- 14
- 15
- 16
- 17
- 18
- 19
- 20
- 21
- 22
- 23
- 24
- 25
- 26
- 27
- 28
- 29
- 30
- 31
- 32
- 33
- 34
- 35
- 36
- 37
- 38
- 39
- 40

=== FILES LOADED SUCCESSFULLY ===
Number of selected recordings: 40
Tracks shape: (25373095, 28)
TracksMeta shape: (68286, 19)
RecordingMeta shape: (40, 17)

=== TRACK FILES USED ===
01_tracks.csv
02_tracks.csv
03_tracks.csv
04_tracks.csv
05_tracks.csv
06_tracks.csv
07_tracks.csv
08_tracks.csv
09_tracks.csv
10_tracks.csv
11_tracks.csv
12_tracks.csv
13_tracks.csv
14_tracks.csv
15_tracks.csv
16_tracks.csv
17_tracks.csv
18_tracks.csv
19_tracks.csv
20_tracks.csv
21_tracks.csv
22_tracks.csv
23_tracks.csv
24_tracks.csv
25_tracks.csv
26_tracks.csv
27_tracks.csv
28_tracks.csv
29_tracks.csv
30_tracks.csv
31_tracks.csv
32_tracks.csv
33_tracks.csv
34_tracks.csv
35_tracks.csv
36_tracks.csv
37_tracks.csv
38_tracks.csv
39_tracks.csv
40_tracks.csv

=== TRACKSMETA FILES USED ===
01_t

# Lane changers cleanup

In [36]:
# Cleanup lane changers

# -------------------------------------------------
# ORIGINAL FRAME SPAN PER EGO VEHICLE
# -------------------------------------------------
ego_frame_span = (
    tracks.groupby("unique_id")["frame"]
    .agg(orig_start_frame="min", orig_end_frame="max")
    .reset_index()
)

# -----------------------------
# STEP 1: Identify lane-stable vehicles
# -----------------------------
lane_nunique = tracks.groupby("unique_id")["laneId"].nunique()
lane_stable_ids = set(lane_nunique[lane_nunique == 1].index)
lane_changer_ids = set(lane_nunique[lane_nunique > 1].index)

total_vehicles = tracks["unique_id"].nunique()
num_stable = len(lane_stable_ids)
num_changers = len(lane_changer_ids)

print("\n=== LANE STABILITY ===")
print("Vehicles total:", total_vehicles)
print("Lane-stable vehicles:", num_stable)
print("Lane-changing vehicles:", num_changers)
print(f"Fraction lane-changing: {num_changers / total_vehicles:.2%}")


=== LANE STABILITY ===
Vehicles total: 68286
Lane-stable vehicles: 60011
Lane-changing vehicles: 8275
Fraction lane-changing: 12.12%


In [37]:
# -----------------------------
# STEP 2: Attach leader lane information
# -----------------------------

leader_lookup = tracks.loc[
    tracks["id"] != 0,
    ["recording_id", "frame", "id", "laneId"]
].rename(
    columns={
        "id": "precedingId",
        "laneId": "precedingLaneId"
    }
)

tracks = tracks.merge(
    leader_lookup,
    on=["recording_id", "frame", "precedingId"],
    how="left",
    validate="many_to_one"
)

print(
    "Rows with leader but missing leader lane:",
    ((tracks["precedingId"] != 0) & tracks["precedingLaneId"].isna()).sum()
)

print("tracks shape after leader lane merge:", tracks.shape)
print(tracks[["recording_id", "frame", "id", "precedingId", "laneId", "precedingLaneId"]].head())

Rows with leader but missing leader lane: 0
tracks shape after leader lane merge: (25373095, 29)
  recording_id  frame  id  precedingId  laneId  precedingLaneId
0           01      1   1            0       5              NaN
1           01      2   1            0       5              NaN
2           01      3   1            0       5              NaN
3           01      4   1            0       5              NaN
4           01      5   1            0       5              NaN


In [38]:
# -----------------------------
# STEP 3: Strict validity mask (frame-level)
# -----------------------------

tracks["preceding_unique_id"] = (
    tracks["recording_id"].astype(str) + "_" + tracks["precedingId"].astype(str)
)

tracks["ego_lane_stable"] = tracks["unique_id"].isin(lane_stable_ids)

tracks["lead_lane_stable"] = (
    (tracks["precedingId"] != 0)
    & tracks["preceding_unique_id"].isin(lane_stable_ids)
)

tracks["leader_exists"] = tracks["precedingId"] != 0
tracks["leader_present_same_frame"] = tracks["precedingLaneId"].notna()
tracks["same_lane_as_leader"] = tracks["laneId"] == tracks["precedingLaneId"]

tracks["valid_cf_frame"] = (
    tracks["ego_lane_stable"]
    & tracks["leader_exists"]
    & tracks["lead_lane_stable"]
    & tracks["leader_present_same_frame"]
    & tracks["same_lane_as_leader"]
)

print("Valid car-following frames:", tracks["valid_cf_frame"].sum())
print("Total frames:", len(tracks))
print("Valid share:", tracks["valid_cf_frame"].mean())

Valid car-following frames: 16228401
Total frames: 25373095
Valid share: 0.6395909131306212


In [39]:
# -----------------------------
# STEP 4: Segment into episodes with constant leader
# -----------------------------

tracks = tracks.sort_values(["unique_id", "frame"]).copy()

valid_change = (
    tracks["valid_cf_frame"]
    != tracks.groupby("unique_id")["valid_cf_frame"].shift(1)
)

leader_change = (
    tracks["preceding_unique_id"]
    != tracks.groupby("unique_id")["preceding_unique_id"].shift(1)
)

tracks["run_break"] = (valid_change | leader_change).fillna(True)
tracks["run_id"] = tracks.groupby("unique_id")["run_break"].cumsum()

df_valid = tracks[tracks["valid_cf_frame"]].copy()

episodes = (
    df_valid.groupby(["unique_id", "run_id"], as_index=False)
    .agg(
        recording_id=("recording_id", "first"),
        id=("id", "first"),
        precedingId=("precedingId", "first"),
        preceding_unique_id=("preceding_unique_id", "first"),
        laneId=("laneId", "first"),
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        n_frames=("frame", "size"),
    )
)

min_frames = int(np.ceil(MIN_EPISODE_SECONDS * DEFAULT_FRAMERATE))
episodes["duration_s"] = episodes["n_frames"] / DEFAULT_FRAMERATE

episodes_strict = episodes[episodes["n_frames"] >= min_frames].copy()

# Create unique episode_id
episodes_strict = episodes_strict.reset_index(drop=True)
episodes_strict["episode_id"] = np.arange(len(episodes_strict))

print("\n=== STRICT EPISODES ===")
print("Min episode seconds:", MIN_EPISODE_SECONDS)
print("Min frames:", min_frames)
print("Episodes found:", len(episodes_strict))
print("Unique egos with episodes:", episodes_strict["unique_id"].nunique())
print(episodes_strict.head())


=== STRICT EPISODES ===
Min episode seconds: 5.0
Min frames: 125
Episodes found: 49811
Unique egos with episodes: 49803
  unique_id  run_id recording_id    id  precedingId preceding_unique_id  \
0     01_10       1           01    10            6                01_6   
1   01_1000       1           01  1000          999              01_999   
2   01_1001       1           01  1001          997              01_997   
3   01_1002       1           01  1002          996              01_996   
4   01_1003       1           01  1003          998              01_998   

   laneId  start_frame  end_frame  n_frames  duration_s  episode_id  
0       6            1        185       185        7.40           0  
1       3        21549      21753       205        8.20           1  
2       6        21565      21943       379       15.16           2  
3       2        21598      21869       272       10.88           3  
4       5        21605      21779       175        7.00           4  


In [40]:
print("valid_cf_frame" in tracks.columns)
print(tracks["valid_cf_frame"].value_counts(dropna=False))

True
valid_cf_frame
True     16228401
False     9144694
Name: count, dtype: int64


In [41]:
# -----------------------------
# STEP 5: Attach episode_id back to per-frame data
# -----------------------------

episode_tracks = tracks.loc[tracks["valid_cf_frame"]].copy()

old_cols = [
    c for c in episode_tracks.columns
    if c in [
        "episode_id",
        "start_frame",
        "end_frame",
        "n_frames",
        "duration_s",
        "split"
    ]
    or c.startswith("episode_id_")
    or c.startswith("split_")
]

if old_cols:
    print("Removing old episode/split columns:", old_cols)
    episode_tracks = episode_tracks.drop(columns=old_cols)

episode_tracks = episode_tracks.merge(
    episodes_strict[
        [
            "unique_id",
            "run_id",
            "episode_id",
            "start_frame",
            "end_frame",
            "n_frames",
            "duration_s"
        ]
    ],
    on=["unique_id", "run_id"],
    how="inner",
    validate="many_to_one"
)

episode_tracks = episode_tracks[
    (episode_tracks["frame"] >= episode_tracks["start_frame"])
    & (episode_tracks["frame"] <= episode_tracks["end_frame"])
].copy()

missing_episode = episode_tracks["episode_id"].isna().sum()

print("\n=== EPISODE TRACKS ===")
print("Rows with missing episode_id:", missing_episode)
print("episode_tracks shape:", episode_tracks.shape)
print("Number of episodes:", episode_tracks["episode_id"].nunique())

assert missing_episode == 0

print(
    episode_tracks[
        ["episode_id", "unique_id", "run_id", "frame", "precedingId", "recording_id"]
    ].head()
)


=== EPISODE TRACKS ===
Rows with missing episode_id: 0
episode_tracks shape: (15832633, 43)
Number of episodes: 49811
   episode_id unique_id  run_id  frame  precedingId recording_id
0           0     01_10       1      1            6           01
1           0     01_10       1      2            6           01
2           0     01_10       1      3            6           01
3           0     01_10       1      4            6           01
4           0     01_10       1      5            6           01


In [42]:
# ============================================================
# STEP 6 — OUTPUT CHECKS
# ============================================================

total_frames = len(tracks)
total_vehicles = tracks["unique_id"].nunique()

kept_frames = len(episode_tracks)
kept_egos = episode_tracks["unique_id"].nunique()

print("\n=== CLEANED FRAME DATA ===")
print("Frames kept:", kept_frames, f"({kept_frames / total_frames:.2%} of all frames)")
print("Unique egos kept:", kept_egos, f"({kept_egos / total_vehicles:.2%} of all vehicles)")
print("Unique leaders in episodes:", episode_tracks["preceding_unique_id"].nunique())

print("\n=== EPISODE DURATION SUMMARY (seconds) ===")
print(episodes_strict["duration_s"].describe().to_string())

print("\nSample episodes:")
show_cols = [
    "episode_id",
    "recording_id",
    "id",
    "unique_id",
    "precedingId",
    "preceding_unique_id",
    "laneId",
    "start_frame",
    "end_frame",
    "duration_s",
    "n_frames"
]

print(episodes_strict[show_cols].head(20).to_string(index=False))


=== CLEANED FRAME DATA ===
Frames kept: 15832633 (62.40% of all frames)
Unique egos kept: 49803 (72.93% of all vehicles)
Unique leaders in episodes: 49803

=== EPISODE DURATION SUMMARY (seconds) ===
count    49811.000000
mean        12.714166
std          6.039404
min          5.000000
25%          9.840000
50%         11.800000
75%         14.280000
max         86.000000

Sample episodes:
 episode_id recording_id   id unique_id  precedingId preceding_unique_id  laneId  start_frame  end_frame  duration_s  n_frames
          0           01   10     01_10            6                01_6       6            1        185        7.40       185
          1           01 1000   01_1000          999              01_999       3        21549      21753        8.20       205
          2           01 1001   01_1001          997              01_997       6        21565      21943       15.16       379
          3           01 1002   01_1002          996              01_996       2        21598     

## Dataset separation - 80-20

##Data Output


In [43]:
# ============================================================
# STEP 7 — SAVE CLEANED DATASET WITH EGO-LEVEL / EPISODE-CONSISTENT SPLIT
# ============================================================

episodes_strict = episodes_strict.copy()
episode_tracks = episode_tracks.copy()

# -------------------------------------------------
# 1) EGO-LEVEL SPLIT
#    all episodes of the same ego vehicle stay together
# -------------------------------------------------

unique_egos = episodes_strict["unique_id"].drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
shuffled_egos = rng.permutation(unique_egos)

n_egos = len(shuffled_egos)
n_train = int(n_egos * TRAIN_FRAC)
n_val = int(n_egos * VAL_FRAC)
n_test = n_egos - n_train - n_val

train_egos = set(shuffled_egos[:n_train])
val_egos = set(shuffled_egos[n_train:n_train + n_val])
test_egos = set(shuffled_egos[n_train + n_val:])

def assign_split(unique_id):
    if unique_id in train_egos:
        return "train"
    elif unique_id in val_egos:
        return "val"
    else:
        return "test"

episodes_strict["split"] = episodes_strict["unique_id"].apply(assign_split)

print("\n=== EGO SPLIT COUNTS ===")
print(episodes_strict["split"].value_counts())

# -------------------------------------------------
# 2) ATTACH SPLIT TO FRAME-LEVEL DATA
#    each frame inherits split from its episode
# -------------------------------------------------

old_split_cols = [
    c for c in episode_tracks.columns
    if c == "split" or c.startswith("split_")
]

if old_split_cols:
    print("Removing old split columns:", old_split_cols)
    episode_tracks = episode_tracks.drop(columns=old_split_cols)

episode_tracks = episode_tracks.merge(
    episodes_strict[["episode_id", "split"]],
    on="episode_id",
    how="left",
    validate="many_to_one"
)

missing_split = episode_tracks["split"].isna().sum()
print("Rows with missing split:", missing_split)
assert missing_split == 0

# -------------------------------------------------
# 3) EGO-LEVEL SUMMARY TABLE
# -------------------------------------------------

ego_split_summary = (
    episodes_strict.groupby("unique_id", as_index=False)
    .agg(
        recording_id=("recording_id", "first"),
        original_id=("id", "first"),
        split=("split", "first"),
        n_episodes=("episode_id", "nunique"),
        total_episode_frames=("n_frames", "sum"),
        total_episode_duration_s=("duration_s", "sum")
    )
)

# -------------------------------------------------
# 4) CREATE SEPARATE DATASETS PER SPLIT
# -------------------------------------------------

episodes_train = episodes_strict[episodes_strict["split"] == "train"].copy()
episodes_val   = episodes_strict[episodes_strict["split"] == "val"].copy()
episodes_test  = episodes_strict[episodes_strict["split"] == "test"].copy()

tracks_train = episode_tracks[episode_tracks["split"] == "train"].copy()
tracks_val   = episode_tracks[episode_tracks["split"] == "val"].copy()
tracks_test  = episode_tracks[episode_tracks["split"] == "test"].copy()

ego_train = ego_split_summary[ego_split_summary["split"] == "train"].copy()
ego_val   = ego_split_summary[ego_split_summary["split"] == "val"].copy()
ego_test  = ego_split_summary[ego_split_summary["split"] == "test"].copy()

# -------------------------------------------------
# 5) SAVE FULL DATASETS
# -------------------------------------------------

episode_tracks.to_csv(
    OUTPUT_DIR / "episode_tracks_clean.csv",
    index=False
)

episodes_strict.to_csv(
    OUTPUT_DIR / "episodes_metadata.csv",
    index=False
)

ego_split_summary.to_csv(
    OUTPUT_DIR / "ego_split_summary.csv",
    index=False
)

# -------------------------------------------------
# 6) SAVE SPLIT-SPECIFIC DATASETS
# -------------------------------------------------

tracks_train.to_csv(OUTPUT_DIR / "episode_tracks_train.csv", index=False)
tracks_val.to_csv(OUTPUT_DIR / "episode_tracks_val.csv", index=False)
tracks_test.to_csv(OUTPUT_DIR / "episode_tracks_test.csv", index=False)

episodes_train.to_csv(OUTPUT_DIR / "episodes_metadata_train.csv", index=False)
episodes_val.to_csv(OUTPUT_DIR / "episodes_metadata_val.csv", index=False)
episodes_test.to_csv(OUTPUT_DIR / "episodes_metadata_test.csv", index=False)

ego_train.to_csv(OUTPUT_DIR / "ego_summary_train.csv", index=False)
ego_val.to_csv(OUTPUT_DIR / "ego_summary_val.csv", index=False)
ego_test.to_csv(OUTPUT_DIR / "ego_summary_test.csv", index=False)

# -------------------------------------------------
# 7) PRINT FINAL SUMMARY
# -------------------------------------------------

print("\n=== SAVED FILES TO ===")
print(OUTPUT_DIR)

print("\nFrame counts:")
print("train:", len(tracks_train))
print("val  :", len(tracks_val))
print("test :", len(tracks_test))

print("\nEpisode counts:")
print("train:", len(episodes_train))
print("val  :", len(episodes_val))
print("test :", len(episodes_test))

print("\nEgo counts:")
print("train:", len(ego_train))
print("val  :", len(ego_val))
print("test :", len(ego_test))


=== EGO SPLIT COUNTS ===
split
train    34870
test      7471
val       7470
Name: count, dtype: int64
Rows with missing split: 0

=== SAVED FILES TO ===
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final

Frame counts:
train: 11094352
val  : 2370107
test : 2368174

Episode counts:
train: 34870
val  : 7470
test : 7471

Ego counts:
train: 34862
val  : 7470
test : 7471


In [44]:
# -------------------------------------------------
# 8) LEAKAGE CHECKS
# -------------------------------------------------

train_ids = set(tracks_train["unique_id"].unique())
val_ids = set(tracks_val["unique_id"].unique())
test_ids = set(tracks_test["unique_id"].unique())

print("\n=== SPLIT LEAKAGE CHECKS ===")
print("Train ∩ Val egos:", len(train_ids & val_ids))
print("Train ∩ Test egos:", len(train_ids & test_ids))
print("Val ∩ Test egos:", len(val_ids & test_ids))

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

episode_split_check = episode_tracks.groupby("episode_id")["split"].nunique()

print("Episodes appearing in more than one split:", (episode_split_check > 1).sum())

assert (episode_split_check <= 1).all()

print("Leakage checks passed.")


=== SPLIT LEAKAGE CHECKS ===
Train ∩ Val egos: 0
Train ∩ Test egos: 0
Val ∩ Test egos: 0
Episodes appearing in more than one split: 0
Leakage checks passed.


In [45]:
# -------------------------------------------------
# 9) SAVE DATA PREPARATION MANIFEST
# -------------------------------------------------

manifest = {
    "raw_data_dir": str(RAW_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "n_recordings": N_RECORDINGS,
    "selected_recording_ids": selected_ids,
    "random_seed": RANDOM_SEED,
    "min_episode_seconds": MIN_EPISODE_SECONDS,
    "default_framerate": DEFAULT_FRAMERATE,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "test_frac": TEST_FRAC,
    "n_raw_frames": int(len(tracks)),
    "n_clean_episode_frames": int(len(episode_tracks)),
    "n_episodes": int(len(episodes_strict)),
    "n_egos": int(ego_split_summary["unique_id"].nunique()),
    "split_frame_counts": episode_tracks["split"].value_counts().to_dict(),
    "split_episode_counts": episodes_strict["split"].value_counts().to_dict(),
    "split_ego_counts": ego_split_summary["split"].value_counts().to_dict(),
}

with open(OUTPUT_DIR / "data_preparation_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:")
print(OUTPUT_DIR / "data_preparation_manifest.json")


Saved manifest:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\data_preparation_manifest.json
